# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Classification.**

My decision is: **which content pages should an editor review first because they show a measurable risk of decline?** The person acting on the output is a content/SEO editor, who can review or refresh the highest-priority pages.

I would frame this as classification because the immediate question is whether a page belongs to a **declining vs. not-declining** group. The model output could then be used as decision-support to prioritize review.

This is not a claim that ML predicts future traffic perfectly. The starter dataset gives us a proxy outcome based on measured changes in impressions. A wrong positive call costs editor time; a missed declining page can mean a missed opportunity to refresh content.

In [ ]:
# Load the starter dataset and check the target balance.
# This cell is for code/checks only.

import os
import subprocess
import pandas as pd

REPO_DIR = "/content/flyrank-ml-internship-starter"
REPO_URL = "https://github.com/enoughsudhanshu/flyrank-ml-internship-starter.git"

# The GitHub repository is not automatically mounted in Colab.
# Clone the user's fork so the required starter dataset is available.
if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", "-q", REPO_URL, REPO_DIR],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Could not clone the repository. Check that the repository is accessible."
            + ("\n" + result.stderr if result.stderr else "")
        )

DATA_PATH = os.path.join(
    REPO_DIR, "data", "raw", "content_refresh_anonymized.csv"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

df["is_declining_proxy"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Proxy positive rate:", round(df["is_declining_proxy"].mean(), 3))
print("Proxy label counts:")
print(df["is_declining_proxy"].value_counts().sort_index())

## 2. Target or proxy

I would use `trend_direction == "down"` as a **proxy target** for the first version.

The label is derived from measured search-impression change: `down` means the recent 30-day impressions are more than 20% below the previous 30-day impressions. So this is a defined label built from an observed metric, not a directly observed future outcome such as “the page will decline next month.”

That distinction matters: I would describe the resulting model as **decision-support for pages showing a measured decline pattern**, not as proof that it predicts future decline.

Also, `trend_direction` and `trend_pct` must not be used as input features because they are the source of the target and would leak the answer.

In [ ]:
# Verify how the proxy target relates to the measured trend.

target_check = (
    df.groupby("trend_direction", dropna=False)
      .agg(
          rows=("content_id", "size"),
          median_trend_pct=("trend_pct", "median"),
          min_trend_pct=("trend_pct", "min"),
          max_trend_pct=("trend_pct", "max"),
      )
      .sort_index()
)

display(target_check)

print(
    "\nProxy positive rate:",
    round(df["is_declining_proxy"].mean(), 3)
)

print(
    "Target-source columns excluded from model features:",
    ["trend_direction", "trend_pct"]
)

## 3. Success metric

**Primary metric: ROC-AUC.**

For this binary classification problem, ROC-AUC measures how well the model separates pages with the declining proxy label from pages without it across probability thresholds.

A score of **0.50 is chance-level**, while **1.00 is perfect separation**. I would treat a result around **0.70 or higher as promising**, but only if it is measured on held-out clients and compared with a simple baseline.

The metric should be chosen before training so that I do not redefine “good” after seeing the results.

Because the real action is prioritizing pages for review, I would also inspect **precision at a fixed review capacity** later; however, ROC-AUC is the single primary metric for this framing.

In [ ]:
# Metric sanity check: reference points.

print("Chance ROC-AUC:", 0.50)
print("Promising reference point for this framing:", 0.70)
print("Starter baseline ROC-AUC reported in the repo:", 0.627)
print("Starter random-forest ROC-AUC reported in the repo:", 0.750)

## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item/page.**

The starter data contains 30,000 content-item rows. For the framing step, I will show a small slice with the identifiers, useful pre-label signals, and the proxy target.

`client_id` is kept only to show the client grouping; it is not a model feature.

In [ ]:
# One row = one pseudonymized content item/page.

lane_slice = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "trend_direction",
        "trend_pct",
        "is_declining_proxy",
    ]
].head(10)

print("Shape of displayed slice:", lane_slice.shape)
display(lane_slice)

## 5. Why ML beats a fixed rule here

A single rule such as **“refresh every page older than 180 days”** is easy to write, but it ignores other signals that may interact: impressions, clicks, sessions, CTR, average position, content age, freshness, content type, search intent, and engagement.

For example, an older page with strong demand may deserve a different decision from an older page with falling impressions and weak engagement. A single threshold cannot easily capture those combinations.

ML is therefore useful **only if a learned model can improve held-out decision quality over the fixed baseline**. The starter results suggest this is plausible: the provided baseline has ROC-AUC 0.627, while the starter random forest reaches 0.750. I would still describe this as evidence from the anonymized starter slice, not a guarantee for new clients or future data.

In [ ]:
# Show that the problem has multiple candidate signals rather than one obvious threshold.
# trend_direction and trend_pct are excluded because they define the proxy target.

signal_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
]

signal_summary = (
    df[signal_columns]
    .describe()
    .T[["count", "mean", "50%", "std"]]
)

display(signal_summary)

print("\nCandidate signals shown:", len(signal_summary))
print(
    "Note: trend_direction and trend_pct are excluded "
    "from model features because they define the proxy target."
)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook uses the starter dataset at `data/raw/content_refresh_anonymized.csv`
- [x] One row is clearly defined as one pseudonymized content item/page
- [x] The proxy target is explained honestly and its leakage columns are excluded from features
- [x] The primary metric is named before training
- [x] Claims are limited to measured/proxy/decision-support language
- [x] No client names, private queries, or client-specific URLs are included
- [ ] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.